IDE3 Algorithm

### ID3 Algorithm Implementation

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter

# Helper function to calculate entropy
def calculate_entropy(data):
    if len(data) == 0:
        return 0

    # Count occurrences of each class label
    class_labels = data.iloc[:, -1] # Assuming last column is the target class
    label_counts = Counter(class_labels)

    entropy = 0
    for label in label_counts:
        probability = label_counts[label] / len(class_labels)
        entropy -= probability * np.log2(probability)
    return entropy

# Helper function to calculate information gain
def calculate_information_gain(data, attribute_name, target_attribute_name):
    total_entropy = calculate_entropy(data[[target_attribute_name]])

    # Calculate entropy for each value of the attribute
    attribute_values = data[attribute_name].unique()
    weighted_average_entropy = 0
    for value in attribute_values:
        subset = data[data[attribute_name] == value]
        probability = len(subset) / len(data)
        weighted_average_entropy += probability * calculate_entropy(subset[[target_attribute_name]])

    information_gain = total_entropy - weighted_average_entropy
    return information_gain

# Node class for the Decision Tree
class Node:
    def __init__(self, attribute=None, value=None, result=None, children=None):
        self.attribute = attribute    # Attribute to split on
        self.value = value        # Value of the parent attribute that leads to this node
        self.result = result      # Class label if it's a leaf node
        self.children = children if children is not None else {} # Dictionary of child nodes

    def __repr__(self):
        if self.result is not None:
            return f"Leaf({self.result})"
        return f"Node({self.attribute}, Children: {list(self.children.keys())})"

# ID3 Algorithm implementation
def id3(data, target_attribute_name, attributes):
    # If all examples have the same target value, return a leaf node with that value
    target_values = data[target_attribute_name].unique()
    if len(target_values) == 1:
        return Node(result=target_values[0])

    # If there are no more attributes to split on, return a leaf node with the most common target value
    if not attributes:
        most_common_target = data[target_attribute_name].mode()[0]
        return Node(result=most_common_target)

    # Select the best attribute to split on (attribute with highest information gain)
    best_attribute = None
    max_info_gain = -1
    for attr in attributes:
        info_gain = calculate_information_gain(data, attr, target_attribute_name)
        if info_gain > max_info_gain:
            max_info_gain = info_gain
            best_attribute = attr

    # Create a new decision node
    node = Node(attribute=best_attribute)

    # Remove the best attribute from the list of available attributes
    remaining_attributes = [attr for attr in attributes if attr != best_attribute]

    # For each value of the best attribute, create a child branch
    for value in data[best_attribute].unique():
        subset = data[data[best_attribute] == value]
        if not subset.empty:
            child_node = id3(subset, target_attribute_name, remaining_attributes)
            child_node.value = value # Store the value that led to this child
            node.children[value] = child_node
        else:
            # If a subset is empty, create a leaf node with the most common target value from the parent
            most_common_target = data[target_attribute_name].mode()[0]
            node.children[value] = Node(result=most_common_target, value=value)

    return node

# Function to predict a class for a new instance
def predict(tree, instance):
    if tree.result is not None:
        return tree.result

    attribute_value = instance[tree.attribute]
    if attribute_value in tree.children:
        return predict(tree.children[attribute_value], instance)
    else:
        # Handle cases where the instance has an unknown attribute value
        # For simplicity, returning None or the most common class from the tree's children
        # In a real scenario, more robust handling would be needed.
        return None


### ID3 Algorithm Example

In [ ]:
# Sample dataset (Weather for playing golf)
data = {
    'Outlook': ['Sunny', 'Sunny', 'Overcast', 'Rain', 'Rain', 'Rain', 'Overcast', 'Sunny', 'Sunny', 'Rain', 'Sunny', 'Overcast', 'Overcast', 'Rain'],
    'Temperature': ['Hot', 'Hot', 'Hot', 'Mild', 'Cool', 'Cool', 'Cool', 'Mild', 'Cool', 'Mild', 'Mild', 'Mild', 'Hot', 'Mild'],
    'Humidity': ['High', 'High', 'High', 'High', 'Normal', 'Normal', 'Normal', 'High', 'Normal', 'Normal', 'Normal', 'High', 'Normal', 'High'],
    'Wind': ['Weak', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 'Weak', 'Strong'],
    'PlayTennis': ['No', 'No', 'Yes', 'Yes', 'Yes', 'No', 'Yes', 'No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No']
}
df = pd.DataFrame(data)

print("Dataset:")
display(df)

target_attribute = 'PlayTennis'
attributes = [col for col in df.columns if col != target_attribute]

# Build the ID3 tree
print("\nBuilding Decision Tree...")
id3_tree = id3(df, target_attribute, attributes)
print("Decision Tree built successfully!")

# Function to print the tree (for visualization)
def print_tree(node, depth=0, branch_value=None):
    indent = "  " * depth
    if branch_value is not None:
        branch_str = f"Branch ({branch_value}): "
    else:
        branch_str = ""

    if node.result is not None:
        print(f"{indent}{branch_str}Predict: {node.result}")
    else:
        print(f"{indent}{branch_str}Split on: {node.attribute}")
        for value, child in node.children.items():
            print_tree(child, depth + 1, value)

print("\nDecision Tree Structure:")
print_tree(id3_tree)

# Make predictions
print("\nMaking Predictions:")
new_instance1 = pd.Series({'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'})
prediction1 = predict(id3_tree, new_instance1)
print(f"Instance: {new_instance1.to_dict()} -> Prediction: {prediction1}")

new_instance2 = pd.Series({'Outlook': 'Rain', 'Temperature': 'Hot', 'Humidity': 'Normal', 'Wind': 'Weak'})
prediction2 = predict(id3_tree, new_instance2)
print(f"Instance: {new_instance2.to_dict()} -> Prediction: {prediction2}")


Dataset:


,Outlook,Temperature,Humidity,Wind,PlayTennis
0,Sunny,Hot,High,Weak,No
1,Sunny,Hot,High,Strong,No
2,Overcast,Hot,High,Weak,Yes
3,Rain,Mild,High,Weak,Yes
4,Rain,Cool,Normal,Weak,Yes
5,Rain,Cool,Normal,Strong,No
6,Overcast,Cool,Normal,Strong,Yes
7,Sunny,Mild,High,Weak,No
8,Sunny,Cool,Normal,Weak,Yes
9,Rain,Mild,Normal,Weak,Yes



Building Decision Tree...
Decision Tree built successfully!

Decision Tree Structure:
Split on: Outlook
  Branch (Sunny): Split on: Humidity
    Branch (High): Predict: No
    Branch (Normal): Predict: Yes
  Branch (Overcast): Predict: Yes
  Branch (Rain): Split on: Wind
    Branch (Weak): Predict: Yes
    Branch (Strong): Predict: No

Making Predictions:
Instance: {'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'} -> Prediction: No
Instance: {'Outlook': 'Rain', 'Temperature': 'Hot', 'Humidity': 'Normal', 'Wind': 'Weak'} -> Prediction: Yes


A* Algorithm

In [ ]:
def a_star_search(graph, start, goal, h_costs):


    # The set of nodes to be evaluated, storing (f_cost, g_cost, node, parent_node).
    # We're simulating a min-priority queue with a list for simplicity.
    open_set = [{'f': h_costs[start], 'g': 0, 'node': start, 'parent': None}]

    # Stores the most efficient way to reach each node found so far:
    # {node: (cost_from_start, parent_node)}
    came_from = {start: (0, None)}

    # Nodes that have already been fully processed.
    closed_set = set()

    while open_set:
        # Pick the node with the lowest f_cost from the open set.
        current_item = min(open_set, key=lambda item: item['f'])
        open_set.remove(current_item)

        current_node = current_item['node']
        current_g_cost = current_item['g']

        # If this node was already processed via a cheaper path, skip it.
        if current_node in closed_set:
            continue

        closed_set.add(current_node)

        # If we reached the goal, reconstruct and return the path.
        if current_node == goal:
            path = []
            n = goal
            while n is not None:
                path.append(n)
                n = came_from[n][1] # Trace back using parent info
            return path[::-1], current_g_cost

        # Explore neighbors of the current node.
        for neighbor, weight in graph[current_node].items():
            if neighbor in closed_set:
                continue

            # Calculate cost to reach the neighbor via the current path.
            tentative_g_cost = current_g_cost + weight

            # If this path to the neighbor is better than a previous one,
            # or if we haven't seen this neighbor before.
            if neighbor not in came_from or tentative_g_cost < came_from[neighbor][0]:
                came_from[neighbor] = (tentative_g_cost, current_node)
                heuristic = h_costs.get(neighbor, 0) # Get heuristic, default to 0 if not set
                new_f_cost = tentative_g_cost + heuristic
                open_set.append({'f': new_f_cost, 'g': tentative_g_cost, 'node': neighbor, 'parent': current_node})

    return None, None # No path found

In [ ]:
print("\nA* Search Example")
# Define a sample graph with edge weights
graph_a_star_ex = {
    'A': {'B': 1, 'C': 4},
    'B': {'D': 2, 'E': 5},
    'C': {'F': 1},
    'D': {'G': 3},
    'E': {'G': 1},
    'F': {'G': 1},
    'G': {}
}

# Define heuristic costs for each node to the goal
heuristic_a_star_ex = {
    'A': 7,
    'B': 6,
    'C': 2,
    'D': 2,
    'E': 1,
    'F': 1,
    'G': 0 # Heuristic for the goal node is always 0
}

start_a_star_ex = 'A'
goal_a_star_ex = 'G'

# Run the A* search
path_a_star, cost_a_star = a_star_search(graph_a_star_ex, start_a_star_ex, goal_a_star_ex, heuristic_a_star_ex)

if path_a_star:
    print(f"A* Path from {start_a_star_ex} to {goal_a_star_ex}: {path_a_star}")
    print(f"A* Total Cost: {cost_a_star}")
else:
    print(f"A* Path from {start_a_star_ex} to {goal_a_star_ex} not found.")


A* Search Example
A* Path from A to G: ['A', 'C', 'F', 'G']
A* Total Cost: 6


Uniform Cost Search

In [ ]:
def uniform_cost_search(graph, start, goal):
    """
    Finds the cheapest path from 'start' to 'goal' using Uniform Cost Search.

    Args:
        graph (dict): Adjacency list, e.g., {node: {neighbor: weight, ...}}.
        start (str): The starting node.
        goal (str): The target node.

    Returns:
        tuple: (list, float) The path as a list of nodes and its total cost.
               Returns (None, None) if no path is found.
    """

    # Nodes to visit, ordered by lowest accumulated cost (g).
    # We're simulating a min-priority queue with a list.
    open_list = [
        {'g': 0, 'node': start, 'parent': None}
    ]

    # Stores the best path to reach each node found so far:
    # {node: (cost_from_start, parent_node)}
    came_from = {start: (0, None)}

    # Nodes that have already been fully processed.
    explored_nodes = set()

    while open_list:
        # Get the node with the lowest path cost from the open list.
        current_node_info = min(open_list, key=lambda item: item['g'])
        open_list.remove(current_node_info)

        current_node = current_node_info['node']
        current_g_cost = current_node_info['g']

        # If we've already found a cheaper path to this node, skip.
        if current_node in explored_nodes:
            continue

        explored_nodes.add(current_node)

        # If we reached the goal, reconstruct and return the path.
        if current_node == goal:
            path = []
            n = goal
            while n is not None:
                path.append(n)
                n = came_from[n][1] # Trace back using parent info
            return path[::-1], current_g_cost

        # Explore neighbors
        for neighbor, weight in graph[current_node].items():
            if neighbor in explored_nodes:
                continue

            tentative_g_cost = current_g_cost + weight

            # If this path to neighbor is better or new, update it.
            if neighbor not in came_from or tentative_g_cost < came_from[neighbor][0]:
                came_from[neighbor] = (tentative_g_cost, current_node)
                open_list.append({'g': tentative_g_cost, 'node': neighbor, 'parent': current_node})

    return None, None # No path found

In [ ]:
print("\nUniform Cost Search Example")
# Define a sample graph with edge weights
graph_ucs_ex = {
    'A': {'B': 1, 'C': 5},
    'B': {'D': 2, 'E': 4},
    'C': {'F': 3},
    'D': {'G': 3},
    'E': {'G': 1},
    'F': {'G': 1},
    'G': {}
}

start_ucs_ex = 'A'
goal_ucs_ex = 'G'

# Run the UCS search
path_ucs, cost_ucs = uniform_cost_search(graph_ucs_ex, start_ucs_ex, goal_ucs_ex)

if path_ucs:
    print(f"UCS Path from {start_ucs_ex} to {goal_ucs_ex}: {path_ucs}")
    print(f"UCS Total Cost: {cost_ucs}")
else:
    print(f"UCS Path from {start_ucs_ex} to {goal_ucs_ex} not found.")


Uniform Cost Search Example
UCS Path from A to G: ['A', 'B', 'D', 'G']
UCS Total Cost: 6


Greedy Best-First Search

In [ ]:
def greedy_best_first_search(graph, start, goal, h_costs):
    """
    Finds a path from 'start' to 'goal' using Greedy Best-First Search,
    prioritizing nodes with the lowest heuristic cost to the goal.

    Args:
        graph (dict): Adjacency list, e.g., {node: {neighbor: weight, ...}}.
        start (str): The starting node.
        goal (str): The target node.
        h_costs (dict): Heuristic cost estimates from each node to the goal.

    Returns:
        tuple: (list, float) The path as a list of nodes and its total cost.
               Returns (None, None) if no path is found.
    """

    # Nodes to visit, ordered solely by heuristic cost (h).
    # We're simulating a min-priority queue with a list.
    open_set_gbfs = [{'h': h_costs[start], 'node': start, 'parent': None, 'cost_so_far': 0}]

    # Stores the parent of each node in the path found so far:
    # {node: (cost_from_start, parent_node)}
    came_from_gbfs = {start: (0, None)}

    # Nodes that have already been fully processed.
    explored_gbfs = set()

    while open_set_gbfs:
        # Get the node with the lowest heuristic cost.
        current_node_item = min(open_set_gbfs, key=lambda item: item['h'])
        open_set_gbfs.remove(current_node_item)

        current_node = current_node_item['node']
        current_path_cost = current_node_item['cost_so_far']

        # If we've already processed this node, skip.
        if current_node in explored_gbfs:
            continue

        explored_gbfs.add(current_node)

        # If we reached the goal, reconstruct and return the path.
        if current_node == goal:
            path = []
            n = goal
            while n is not None:
                path.append(n)
                n = came_from_gbfs[n][1] # Trace back using parent info
            return path[::-1], current_path_cost

        # Explore neighbors
        for neighbor, weight in graph[current_node].items():
            if neighbor in explored_gbfs:
                continue

            # For GBFS, we're primarily guided by the heuristic. The `cost_so_far`
            # helps in accurately reporting the path cost once the goal is found.
            tentative_total_cost = current_path_cost + weight

            # Update path information if we find a new or cheaper way to this neighbor.
            if neighbor not in came_from_gbfs or tentative_total_cost < came_from_gbfs[neighbor][0]:
                came_from_gbfs[neighbor] = (tentative_total_cost, current_node)
                # Add to open_set based on its heuristic value.
                heuristic = h_costs.get(neighbor, 0)
                open_set_gbfs.append({'h': heuristic, 'node': neighbor, 'parent': current_node, 'cost_so_far': tentative_total_cost})

    return None, None # No path found

In [ ]:
print("\nGreedy Best-First Search Example")
# Define a sample graph with edge weights
graph_gbfs_ex = {
    'A': {'B': 1, 'C': 10},
    'B': {'D': 1, 'E': 1},
    'C': {'F': 1},
    'D': {'G': 1},
    'E': {'G': 1},
    'F': {'G': 1},
    'G': {}
}

# Define heuristic costs for each node to the goal
heuristic_gbfs_ex = {
    'A': 5,
    'B': 4,
    'C': 2,
    'D': 3,
    'E': 1,
    'F': 1,
    'G': 0
}

start_gbfs_ex = 'A'
goal_gbfs_ex = 'G'

# Run the Greedy Best-First Search
path_gbfs, cost_gbfs = greedy_best_first_search(graph_gbfs_ex, start_gbfs_ex, goal_gbfs_ex, heuristic_gbfs_ex)

if path_gbfs:
    print(f"GBFS Path from {start_gbfs_ex} to {goal_gbfs_ex}: {path_gbfs}")
    print(f"GBFS Total Cost: {cost_gbfs}")
else:
    print(f"GBFS Path from {start_gbfs_ex} to {goal_gbfs_ex} not found.")


Greedy Best-First Search Example
GBFS Path from A to G: ['A', 'C', 'F', 'G']
GBFS Total Cost: 12


BFS

In [ ]:
from collections import deque

def breadth_first_search(graph, start, goal):
    """
    Finds the shortest path in an unweighted graph, or a path with minimal
    edge count in a weighted graph, using Breadth-First Search (BFS).

    Args:
        graph (dict): Adjacency list, e.g., {node: {neighbor: weight, ...}}.
        start (str): The starting node.
        goal (str): The target node.

    Returns:
        tuple: (list, float) The path as a list of nodes and its total cost.
               Returns (None, None) if no path is found.
    """
    # A queue to store (current_node, current_path, accumulated_cost)
    q = deque([(start, [start], 0)])
    visited_nodes = set()

    while q:
        current_node, current_path, current_cost = q.popleft()

        # If we reached the goal, return the path and its cost.
        if current_node == goal:
            return current_path, current_cost

        # Process node if not already visited.
        if current_node not in visited_nodes:
            visited_nodes.add(current_node)

            # Explore neighbors
            for neighbor, weight in graph[current_node].items():
                if neighbor not in visited_nodes:
                    # Create a new path for the neighbor.
                    new_path = list(current_path)
                    new_path.append(neighbor)
                    new_cost = current_cost + weight
                    q.append((neighbor, new_path, new_cost))
    return None, None # No path found

In [ ]:
print("\nBreadth-First Search Example")
# Define a sample graph with edge weights
graph_bfs_ex = {
    'A': {'B': 1, 'C': 1},
    'B': {'D': 1, 'E': 1},
    'C': {'F': 1},
    'D': {'G': 1},
    'E': {'G': 1},
    'F': {'G': 1},
    'G': {}
}

start_bfs_ex = 'A'
goal_bfs_ex = 'G'

# Run the BFS search
path_bfs, cost_bfs = breadth_first_search(graph_bfs_ex, start_bfs_ex, goal_bfs_ex)

if path_bfs:
    print(f"BFS Path from {start_bfs_ex} to {goal_bfs_ex}: {path_bfs}")
    print(f"BFS Total Cost: {cost_bfs}")
else:
    print(f"BFS Path from {start_bfs_ex} to {goal_bfs_ex} not found.")


Breadth-First Search Example
BFS Path from A to G: ['A', 'B', 'D', 'G']
BFS Total Cost: 3


## Depth-First Search

In [ ]:
def depth_first_search(graph, start, goal):
    """
    Finds a path from 'start' to 'goal' using Depth-First Search (DFS).
    Note: DFS does not guarantee the shortest path or lowest cost path.

    Args:
        graph (dict): Adjacency list, e.g., {node: {neighbor: weight, ...}}.
        start (str): The starting node.
        goal (str): The target node.

    Returns:
        tuple: (list, float) The path as a list of nodes and its total cost.
               Returns (None, None) if no path is found.
    """
    # A stack to store (current_node, current_path, accumulated_cost)
    s = [(start, [start], 0)]
    visited_dfs = set()

    while s:
        current_node, current_path, current_cost = s.pop()

        # If we reached the goal, return the path and its cost.
        if current_node == goal:
            return current_path, current_cost

        # Process node if not already visited.
        if current_node not in visited_dfs:
            visited_dfs.add(current_node)

            # Explore neighbors. Push them in reverse order to explore
            # in a somewhat consistent 'left-to-right' manner from graph definition.
            for neighbor, weight in reversed(list(graph[current_node].items())):
                if neighbor not in visited_dfs:
                    # Create a new path for the neighbor.
                    new_path = list(current_path)
                    new_path.append(neighbor)
                    new_cost = current_cost + weight
                    s.append((neighbor, new_path, new_cost))
    return None, None # No path found

In [ ]:
print("\nDepth-First Search Example")
# Define a sample graph with edge weights
graph_dfs_ex = {
    'A': {'B': 1, 'C': 1},
    'B': {'D': 1, 'E': 1},
    'C': {'F': 1},
    'D': {'G': 1},
    'E': {'G': 1},
    'F': {'G': 1},
    'G': {}
}

start_dfs_ex = 'A'
goal_dfs_ex = 'G'

# Run the DFS search
path_dfs, cost_dfs = depth_first_search(graph_dfs_ex, start_dfs_ex, goal_dfs_ex)

if path_dfs:
    print(f"DFS Path from {start_dfs_ex} to {goal_dfs_ex}: {path_dfs}")
    print(f"DFS Total Cost: {cost_dfs}")
else:
    print(f"DFS Path from {start_dfs_ex} to {goal_dfs_ex} not found.")


Depth-First Search Example
DFS Path from A to G: ['A', 'B', 'D', 'G']
DFS Total Cost: 3
